<a href="https://colab.research.google.com/github/rohitkayastha200-source/Zebrafish-AI/blob/main/ZebraFish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# zebrafish_brain_ai.py
# Requirements: gymnasium, torch, numpy (pip install gymnasium torch numpy)

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# -- 1. Environment stub (2D, Gymnasium API) --
class ZebrafishEnv(gym.Env):
    def __init__(self):
        super().__init__()
        # Discrete bout actions: 0=idle,1=forward,2=left,3=right
        self.action_space = gym.spaces.Discrete(4)
        # Observation: dict of vision (1x64x64), flow (8,), odor (1,), drives (2,)
        self.observation_space = gym.spaces.Dict({
            'vision': gym.spaces.Box(low=0, high=1, shape=(1,64,64), dtype=np.float32),
            'flow':   gym.spaces.Box(low=-1,high=1, shape=(8,), dtype=np.float32),
            'odor':   gym.spaces.Box(low=0, high=1, shape=(1,), dtype=np.float32),
            'drives': gym.spaces.Box(low=0, high=100, shape=(2,), dtype=np.float32)
        })
        self.state = None

    def reset(self):
        # Initialize state with random vision/odor, zero flow, zero drives
        vision = np.random.rand(1,64,64).astype(np.float32)
        flow = np.zeros(8, dtype=np.float32)
        odor = np.array([0.0], dtype=np.float32)
        drives = np.array([0.0, 0.0], dtype=np.float32)  # [hunger, fear]
        self.state = (vision, flow, odor, drives)
        return self.state, {}

    def step(self, action):
        vision, flow, odor, drives = self.state
        # Simulate world update (stub)
        # Random new vision (e.g. food moves), static flow, odor increases if forward
        vision = np.random.rand(1,64,64).astype(np.float32)
        flow = np.zeros_like(flow)
        odor = np.clip(odor + 0.05*(action==1), 0.0, 1.0)  # forward increases odor (food)
        # Update drives: hunger rises, fear if predator (none here)
        drives[0] += 1.0       # hunger increases each step
        drives[1] = max(drives[1] - 0.1, 0.0)  # fear decays
        # Reward: +1 for eating (if odor high and action forward), else 0
        reward = 0.0
        if action == 1 and odor[0] > 0.5:
            reward = 1.0
            odor[0] = 0.0  # consumed food
        terminated = False  # no terminal condition in stub
        truncated = False
        self.state = (vision, flow, odor, drives)
        return self.state, reward, terminated, truncated, {}

# -- 2. Sensor encoders --
class VisionEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, stride=2), nn.ReLU(),  # ->16x30x30
            nn.Conv2d(16, 32, 5, 2), nn.ReLU(),                    # ->32x13x13
            nn.Conv2d(32, 64, 3, 2), nn.ReLU()                     # ->64x6x6
        )
        self.fc = nn.Linear(64*6*6, 128)
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return torch.relu(self.fc(x))  # 128-dim feature

class FlowEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(8,32), nn.ReLU(), nn.Linear(32,16), nn.ReLU())
    def forward(self, x):
        return self.net(x)  # 16-dim

class OdorEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1,16), nn.ReLU())
    def forward(self, x):
        return self.net(x)  # 16-dim

# -- 3. RNN-based policy (Actor-Critic) --
class BrainNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=128+16+16+2, hidden_size=128, batch_first=True)
        self.actor = nn.Linear(128, 4)
        self.critic = nn.Linear(128, 1)
    def forward(self, x, hx):
        # x: [batch,1,dim], hx: (h,c)
        out, hx = self.lstm(x, hx)
        h = out[:, -1, :]  # last output
        return self.actor(h), self.critic(h), hx

# -- 4. Intrinsic curiosity (forward model) --
class ForwardModel(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 64), nn.ReLU(),
            nn.Linear(64, state_dim)
        )
    def forward(self, s, a):
        return self.net(torch.cat([s,a], dim=-1))

# -- 5. Agent integrating modules --
class FishAgent:
    def __init__(self):
        super().__init__()
        self.vision_enc = VisionEncoder()
        self.flow_enc = FlowEncoder()
        self.odor_enc = OdorEncoder()
        self.brain = BrainNet()
        # Forward model: predict next latent state (128+16+16+2=162)
        self.forward = ForwardModel(162, 4)
        # Optimizers
        self.opt = optim.Adam(self.parameters(), lr=3e-4)
        self.opt_forward = optim.Adam(self.forward.parameters(), lr=3e-4)

    def parameters(self):
        return list(self.vision_enc.parameters()) \
             + list(self.flow_enc.parameters()) \
             + list(self.odor_enc.parameters()) \
             + list(self.brain.parameters())

    def encode_state(self, obs):
        vision = torch.tensor(obs[0], dtype=torch.float32).unsqueeze(0)
        flow   = torch.tensor(obs[1], dtype=torch.float32).unsqueeze(0)
        odor   = torch.tensor(obs[2], dtype=torch.float32).unsqueeze(0)
        drives = torch.tensor(obs[3], dtype=torch.float32).unsqueeze(0)

        vfeat = self.vision_enc(vision)
        ffeat = self.flow_enc(flow)
        ofeat = self.odor_enc(odor)

        return torch.cat([vfeat, ffeat, ofeat, drives], dim=1)

    def select_action(self, obs, hx):
        # Preprocess obs into tensors and encode sensors
        features = self.encode_state(obs).unsqueeze(1)

        logits, value, hx = self.brain(features, hx)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action.item(), dist.log_prob(action), value, hx, features.squeeze(1)

    def update(self):
        # Placeholder for PPO update: compute losses, backprop
        pass

# -- 6. Training loop (skeleton) --
def train(agent, env, episodes=5, max_steps=100):
    gamma = 0.99
    for ep in range(episodes):
        obs, _ = env.reset()
        hx = (torch.zeros(1,1,128), torch.zeros(1,1,128))
        total_reward = 0.0
        for t in range(max_steps):
            action, logp, value, hx, state_vec = agent.select_action(obs, hx)
            next_obs, ext_reward, terminated, truncated, _ = env.step(action)
            # Intrinsic reward: forward model prediction error
            a_onehot = torch.zeros(1,4); a_onehot[0,action]=1.0
            pred_next = agent.forward(state_vec, a_onehot)
            # Build next state_vec by encoding next_obs
            next_state_vec = agent.encode_state(next_obs)
            intrinsic = torch.norm(pred_next - next_state_vec.detach()).item()
            reward = ext_reward + 0.1 * intrinsic
            total_reward += reward
            # PPO update would occur here (omitted for brevity)
            obs = next_obs
            if terminated or truncated: break
        print(f"Episode {ep+1}: Reward={total_reward:.2f}")
    print("Training finished.")

# -- 7. Run example --
if __name__ == "__main__":
    env = ZebrafishEnv()
    agent = FishAgent()
    train(agent, env, episodes=10, max_steps=50)


Episode 1: Reward=137.29
Episode 2: Reward=136.27
Episode 3: Reward=136.30
Episode 4: Reward=136.27
Episode 5: Reward=136.29
Episode 6: Reward=136.28
Episode 7: Reward=136.31
Episode 8: Reward=136.30
Episode 9: Reward=136.30
Episode 10: Reward=136.28
Training finished.
